# IndicTrans2 adaptation · explicit language tags

An **optional, separate Kaggle/Colab notebook** for native Maithili/Nepali or experimental prefix/proxy conditioning for Bhojpuri, Magahi, Awadhi, Braj and Marwari. Use **real, licensed** parallel JSONL with `source_lang` and `target_lang` fields; never write tags into raw references. Only a checkpoint adapted using the same conditioning strategy should be evaluated as adapted. Native IT2 preprocessing is retained through `IndicProcessor`; custom Hugging Face model code must be explicitly approved.

Separate notebooks are intentional because IndicTrans2, NLLB and BhashaVerse have different tokenizer/runtime stacks. See `docs/multilingual-adaptation.md` before enabling training.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# CPU corpus inspection is the default. Switch to train / train,repairs
# with a GPU and a fresh kernel before running heavyweight model cells.
SETUP_MODE = "base"  # base | preflight | train | train,repairs
REQUIRE_CUDA = SETUP_MODE.startswith("train")
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
ROOT = Path("/kaggle/working/BhaashikMTFinetuning" if Path("/kaggle/working").exists()
            else "/content/BhaashikMTFinetuning" if Path("/content").exists()
            else Path.cwd())
if not (ROOT / "pyproject.toml").is_file():
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git", str(ROOT)], check=True)
os.chdir(ROOT)
installer = ROOT / "scripts" / "setup_notebook.py"
assert installer.is_file(), "Refresh an outdated repository checkout before setup."
loaded = [name for name in ("torch", "transformers", "sentencepiece", "peft", "datasets")
          if name in sys.modules]
if loaded:
    raise RuntimeError("Restart the notebook kernel before installing new dependencies: "
                       + ", ".join(loaded))
command = [sys.executable, str(installer), "--extras", SETUP_MODE]
if REQUIRE_CUDA:
    command.append("--require-cuda")
subprocess.run(command, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.multilingual import require_routes, inspect_routes, write_tagged_copy
from bhaashik.pretrain_review import inspect_splits, plot_review, compare_dev
print("Repo:", ROOT, "Setup mode:", SETUP_MODE)
# The official IndicTransToolkit is a separate dependency not included in
# Bhaashik's NLLB-focused training extras. Install it from a trusted upstream
# source in a fresh model runtime before selecting IT2 inference/training.


## Select actual language pair, strategy and data

Choose native only for supported IndicTrans2 language pairs (e.g. Maithili–Hindi, Nepali–Hindi). `text_prefix` and `dedicated_token` use explicit *supported proxy languages* for upstream IndicProcessor normalization. Proxy routing is a **controlled research ablation**, not native IT2 support.

For unsupported targets such as Hindi → Braj, do not trust the proxy target postprocessor; the adapter saves raw generated text and requires independent evaluation.


In [ ]:
from bhaashik.language_support import language_options
print("IndicTrans2 native language coverage for the seven focus languages:")
for item in language_options():
    print(item["name"], item["corpus_tag"], item["indictrans2"])


In [ ]:
SRC, TGT = "mai_Deva", "hin_Deva"
PAIR = f"{SRC.split('_')[0]}-{TGT.split('_')[0]}"
DATA = ROOT / "data" / "local" / PAIR
MODE = "lora"  # lora | full; dedicated_token requires full
STRATEGY = "native"  # native | text_prefix | dedicated_token
PROXY_SRC = None  # e.g. "hin_Deva" for experimental Braj source
PROXY_TGT = None  # e.g. "hin_Deva" for unsupported Braj target
SEED = 42
RUN = ROOT / "runs" / f"indictrans2-{PAIR}-{STRATEGY}-{MODE}-s{SEED}"
REPORTS = ROOT / "runs" / f"indictrans2-{PAIR}-{STRATEGY}-review"
MODEL = None  # resolved to official direction-specific distilled checkpoint
for split in ("train", "dev", "test"):
    assert (DATA / f"{split}.jsonl").is_file(), f"Supply real {DATA / (split+'.jsonl')}"
    rows, routes = require_routes(DATA / f"{split}.jsonl")
    assert all("source_lang" in r for r in rows), "Migrate legacy splits with write_tagged_copy before training."
    print(split, len(rows), routes)
CORPUS = inspect_splits({split: DATA / f"{split}.jsonl" for split in ("train","dev","test")}, REPORTS)
ROUTES = inspect_routes({split: DATA / f"{split}.jsonl" for split in ("train","dev","test")},
                        REPORTS / "language_routes.json")
print("Ready for structural review:", CORPUS["ready_for_training"])
for issue in CORPUS["issues"]:
    print(issue["severity"], issue["message"])
for chart in plot_review(REPORTS):
    print(chart)


## Optional native IT2 development baseline (before finetuning)

The frozen development sample is shared across models. Original IT2 evaluation uses only **native supported tags**. Do not run this cell for unsupported languages. Other model outputs may be added later using the exact same `sample.jsonl` IDs.


In [ ]:
from bhaashik.model_comparisons import select_dev_sample, run_indictrans2_dev
from bhaashik.indictrans2_adaptation import processing_pair
REPORTS.mkdir(parents=True, exist_ok=True)
DEV_SAMPLE = REPORTS / "sample.jsonl"
if not DEV_SAMPLE.exists():
    select_dev_sample(DATA / "dev.jsonl", DEV_SAMPLE, sample_size=100, seed=SEED)
processing_pair(SRC, TGT, strategy=STRATEGY, proxy_src=PROXY_SRC, proxy_tgt=PROXY_TGT)
RUN_NATIVE_BASELINE = False
DEV_PREDICTIONS = {}
if RUN_NATIVE_BASELINE:
    assert SETUP_MODE.startswith("train"), "Use training extras before loading IndicTrans2."
    assert STRATEGY == "native", "No unfinetuned native baseline for an unsupported language."
    baseline = REPORTS / "indictrans2-original-dev.jsonl"
    if not baseline.exists():
        run_indictrans2_dev(DEV_SAMPLE, baseline, SRC, TGT, batch_size=4,
                           allow_remote_code=True)
    DEV_PREDICTIONS["IndicTrans2-original"] = baseline


## Train (explicit opt-in)

Install the upstream toolkit and the pinned Transformers/PyTorch training stack in a **fresh** runtime first. Confirm the baseline, data licenses, scripts, proxies and the approved upstream custom code. The first pilot may use `max_steps=1`. Full fine-tuning can require more memory than free Kaggle GPUs.


In [ ]:
from bhaashik.indictrans2_adaptation import train_indictrans2
APPROVE_TRAINING = False
ALLOW_REMOTE_MODEL_CODE = False
if APPROVE_TRAINING:
    assert SETUP_MODE.startswith("train"), "Use train or train,repairs setup with a GPU."
    assert ALLOW_REMOTE_MODEL_CODE, "Review upstream custom model code before enabling."
    assert CORPUS["ready_for_training"], "Resolve structural corpus blocking issues."
    config = {
        "train_file": str(DATA / "train.jsonl"), "dev_file": str(DATA / "dev.jsonl"),
        "output_dir": str(RUN), "model": MODEL, "mode": MODE, "strategy": STRATEGY,
        "proxy_src": PROXY_SRC, "proxy_tgt": PROXY_TGT,
        "allow_remote_code": True, "seed": SEED, "epochs": 3,
        "batch_size": 1, "gradient_accumulation_steps": 16,
        "max_length": 128, "max_steps": -1, "precision": "fp16",
    }
    cfg_path = ROOT / "configs" / f"indictrans2-{PAIR}-{STRATEGY}-{MODE}-s{SEED}.json"
    cfg_path.parent.mkdir(parents=True, exist_ok=True)
    cfg_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    print(train_indictrans2(cfg_path))
else:
    print("Training disabled. Review the corpus and models first.")


## Post-training development and held-out test

For a trained checkpoint, produce development predictions and compare on aligned IDs. The test split is used only **after** model selection. Raw text is never mutated with language prefixes.


In [ ]:
from bhaashik.indictrans2_adaptation import generate_indictrans2
from bhaashik.evaluation import evaluate, export_errors
RUN_POSTTRAIN = False
if RUN_POSTTRAIN:
    assert (RUN / "run_manifest.json").is_file(), "Train or attach a reviewed checkpoint first."
    adapted = REPORTS / "indictrans2-adapted-dev.jsonl"
    if not adapted.exists():
        generate_indictrans2(RUN, DEV_SAMPLE, adapted, strategy=STRATEGY,
                            proxy_src=PROXY_SRC, proxy_tgt=PROXY_TGT,
                            allow_remote_code=True)
    DEV_PREDICTIONS["IndicTrans2-adapted"] = adapted
    print(compare_dev(DEV_SAMPLE, DEV_PREDICTIONS, REPORTS))
    test_preds = RUN / "predictions.jsonl"
    if not test_preds.exists():
        generate_indictrans2(RUN, DATA / "test.jsonl", test_preds,
                            strategy=STRATEGY, proxy_src=PROXY_SRC, proxy_tgt=PROXY_TGT,
                            allow_remote_code=True)
    print(evaluate(test_preds, RUN / "evaluation.json"))
    export_errors(test_preds, RUN / "annotations.csv")
else:
    print("Post-training evaluation disabled.")


## Preserve only current experiment

This fast ZIP includes selected run weights, metrics, review plots and language-route diagnostics. For an exact resume, set `INCLUDE_CHECKPOINTS=True`; otherwise intermediate checkpoints are excluded.


In [ ]:
from bhaashik.artifacts import archive_experiment
PRESERVE_ARTIFACTS = False
INCLUDE_CHECKPOINTS = False
if PRESERVE_ARTIFACTS:
    assert RUN.is_dir() and (RUN / "run_manifest.json").exists()
    archive = archive_experiment(
        ROOT, RUN, ROOT.parent / f"indictrans2-{PAIR}-{STRATEGY}-{MODE}-s{SEED}.zip",
        report_dir=REPORTS,
        include_checkpoints=INCLUDE_CHECKPOINTS,
    )
    print("Preserve this archive before stopping the runtime:", archive)
else:
    print("Archiving disabled.")
